# ChemBreak37 | GitHub-to-Colab run

The source of truth is the `chembreak37` directory committed to the ChemBreak GitHub repository.
GPT is the primary cumulative CHCS judge; the local thinking-enabled Qwen model is called only when GPT cannot produce a valid judgment.

Start with `DRY_RUN=True`, use a new run ID and a fresh Colab runtime for live execution, and protect private output files.
This release uses **CB37P-0001..0028** benchmark assignment IDs. The original benchmark behavior text is unchanged.
Renaming task IDs changes the task-derived random seeds relative to archival runs; compare judges on identical saved responses rather than treating two live runs as seed-matched.


In [ ]:
# Cell 2: run controls
DRY_RUN = True                 # Set False for the live experiment
RUN_ID = None                  # None creates cb37_dry_seed_34034 or cb37_live_seed_34034
SEED = 34034
TASK_LIMIT = None              # None means all 28 tasks
PROBE_TOKENIZERS = False       # Optional. True downloads and checks the selected tokenizer in preflight
PROBE_ROLES = False            # Optional. True makes safe test calls to the planner, gate, and GPT and Qwen fallback on a BENIGN probe only
SHOW_LIVE_TEXT = True          # Print source tasks, accepted candidates, and target responses during the run
LIVE_TEXT_LIMIT = 1500         # Maximum displayed characters for each text block

REPOSITORY_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPOSITORY_BRANCH = "main"
PROJECT_SUBDIR = "chembreak37"

# Judge loading only. Keep at 8 layers on GPU initially to leave space for
# the full-size ChemDFM/ChemLLM target on the same A100. Increase only after
# checking nvidia-smi free VRAM. 0 means CPU-only inference (slower).
QWEN_JUDGE_GPU_LAYERS = 8
QWEN_JUDGE_CONTEXT_TOKENS = 16384


In [ ]:
# Cell 3: always fetch the committed ChemBreak37 source from GitHub
from pathlib import Path
import subprocess

REPO_ROOT = Path('/content/chembreak37_repo')
if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists():
        raise RuntimeError(f'{REPO_ROOT} exists but is not a Git checkout. Use a fresh runtime.')
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPOSITORY_BRANCH,
        REPOSITORY_URL, str(REPO_ROOT),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
PROJECT_ROOT = (REPO_ROOT / PROJECT_SUBDIR).resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'ChemBreak37 not found at {PROJECT_ROOT}. Commit the extracted '
        'chembreak37 folder to the configured GitHub branch first.'
    )
import tomllib
installed_meta = tomllib.loads((PROJECT_ROOT / 'pyproject.toml').read_text())['project']
if installed_meta.get('name') != 'chembreak37' or installed_meta.get('version') != '37.0.0':
    raise RuntimeError(f'Expected ChemBreak37 v37.0.0, found {installed_meta}.')
print('ChemBreak37 source:', PROJECT_ROOT)
print('Git commit:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'],text=True).strip())


In [ ]:
# Cell 4: install dependencies and activate this independent package
import subprocess, sys, importlib, site

def run_install(args):
    print('RUNNING:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)

run_install([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = PROJECT_ROOT / 'requirements-cloud-ml.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing dependency file: {requirements}')
run_install([sys.executable, '-m', 'pip', 'install', '-r', requirements])
run_install([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
# Editable installs update a .pth file that a running kernel may not reload.
# Activate the local src tree explicitly so this cell works without a runtime restart.
SRC_ROOT = (PROJECT_ROOT / 'src').resolve()
PACKAGE_ROOT = SRC_ROOT / 'chembreak37'
if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(f'ChemBreak37 source package not found: {PACKAGE_ROOT}')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
site.addsitedir(str(SRC_ROOT))
for module_name in list(sys.modules):
    if module_name == 'chembreak37' or module_name.startswith('chembreak37.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import chembreak37
loaded_from = Path(chembreak37.__file__).resolve()
if PACKAGE_ROOT not in loaded_from.parents:
    raise RuntimeError(f'ChemBreak37 imported from the wrong location: {loaded_from}')
print('ChemBreak37 import OK:', chembreak37.__version__, loaded_from)


In [ ]:
# Cell 5: GPU and persistent working storage
from pathlib import Path
import torch

STORAGE_ROOT = Path('/content/chembreak37_storage').resolve()
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print('STORAGE_ROOT =', STORAGE_ROOT)
print('CUDA available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
    print('bfloat16 supported =', torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown')
elif not DRY_RUN:
    raise RuntimeError('A live ChemBreak37 target run requires a CUDA GPU runtime.')


In [ ]:
# Cell 6: configure credentials for GPT primary and Vertex planner/gate
import os, getpass
if not DRY_RUN:
    if not os.environ.get('GOOGLE_CLOUD_PROJECT', '').strip():
        os.environ['GOOGLE_CLOUD_PROJECT'] = input('Google Cloud project ID: ').strip()
    if not os.environ.get('OPENAI_API_KEY', '').strip():
        os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (hidden input): ').strip()
    if not os.environ['GOOGLE_CLOUD_PROJECT'] or not os.environ['OPENAI_API_KEY']:
        raise RuntimeError('Both GOOGLE_CLOUD_PROJECT and OPENAI_API_KEY are required in live mode')
print('Credentials configured. Live mode =', not DRY_RUN)


In [ ]:
# Cell 7: create an isolated runtime configuration
import copy, yaml

RUN_MODE = 'dry' if DRY_RUN else 'live'
resolved_run_id = RUN_ID or f'cb37_{RUN_MODE}_seed_{SEED}'
base_config = PROJECT_ROOT / 'configs' / 'config.cb37.yaml'
config = yaml.safe_load(base_config.read_text())
config['run'].update({
    'seed': int(SEED),
    'run_mode': RUN_MODE,
    'run_id': resolved_run_id,
    'dry_run': bool(DRY_RUN),
    'task_limit': TASK_LIMIT,
    'live_progress': True,
    'live_progress_show_text': bool(SHOW_LIVE_TEXT),
    'live_progress_text_limit': int(LIVE_TEXT_LIMIT),
    'prompts_path': str(PROJECT_ROOT / 'data' / 'prompts.csv'),
    'manifest_path': str(PROJECT_ROOT / 'data' / 'CB37_prompts28_manifest_v1.csv'),
    'lock_path': str(PROJECT_ROOT / 'data' / 'CB37_prompts28_lock_v1.json'),
    'output_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'runs'),
    'artifact_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'task_artifacts'),
})
config['roles']['chcs_fallback_judge_llm']['model_path'] = str(STORAGE_ROOT / 'models' / 'Qwen3.6-35B-A3B-Q4_K_M.gguf')
for role_name in ('attack_planner_llm', 'intent_gate_llm'):
    config['roles'][role_name]['seed'] = int(SEED)
for target in config['targets']:
    target['cache_dir'] = str(STORAGE_ROOT / 'cache' / 'huggingface' / 'hub')
    target['offload_folder'] = str(STORAGE_ROOT / 'offload' / target['id'])

runtime_config_dir = STORAGE_ROOT / RUN_MODE / resolved_run_id
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config = runtime_config_dir / 'config.cb37.runtime.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print('RUN MODE =', RUN_MODE)
print('RUN ID =', resolved_run_id)
print('RUNTIME CONFIG =', runtime_config)


In [ ]:
# Cell 8: download Qwen3.6-35B-A3B 4-bit ONTO /content disk and start local FALLBACK judge
# Run once per new cloud runtime, BEFORE the live preflight or either target.
# Skip download and build in mock-only DRY_RUN mode.
import os, shutil, subprocess, time, urllib.request, json
from pathlib import Path

QWEN_REPO = 'ggml-org/Qwen3.6-35B-A3B-GGUF'
QWEN_FILE = 'Qwen3.6-35B-A3B-Q4_K_M.gguf'
QWEN_MODEL_DIR = STORAGE_ROOT / 'models'
QWEN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
QWEN_LOCAL_PATH = QWEN_MODEL_DIR / QWEN_FILE
LLAMA_CPP_ROOT = STORAGE_ROOT / 'tools' / 'llama.cpp'
LLAMA_CPP_BIN = LLAMA_CPP_ROOT / 'build' / 'bin' / 'llama-server'
QWEN_LOG_PATH = STORAGE_ROOT / 'tools' / 'llama_cpp_judge.log'
QWEN_PORT = 18037

if DRY_RUN:
    print('DRY_RUN: local 4-bit Qwen model download and server skipped.')
else:
    from huggingface_hub import hf_hub_download
    import torch
    if not QWEN_LOCAL_PATH.exists() or QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        print('Downloading 4-bit GGUF directly to:', QWEN_LOCAL_PATH, flush=True)
        hf_hub_download(repo_id=QWEN_REPO, filename=QWEN_FILE,
                        local_dir=str(QWEN_MODEL_DIR))
    if QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        raise RuntimeError('4-bit model file is incomplete. Rerun this cell to resume.')
    print('4-bit model on /content:', QWEN_LOCAL_PATH,
          '| GB:', round(QWEN_LOCAL_PATH.stat().st_size / 10**9, 2))

    if not LLAMA_CPP_BIN.is_file():
        print('Building CUDA llama.cpp. Compiler output follows...', flush=True)
        LLAMA_CPP_ROOT.parent.mkdir(parents=True, exist_ok=True)
        if not (LLAMA_CPP_ROOT / '.git').is_dir():
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_CPP_ROOT)], check=True)
        cmake_cmd = ['cmake', '-S', str(LLAMA_CPP_ROOT), '-B', str(LLAMA_CPP_ROOT / 'build'),
                     '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release']
        subprocess.run(cmake_cmd, check=True)
        subprocess.run(['cmake', '--build', str(LLAMA_CPP_ROOT / 'build'),
                        '--config', 'Release', '-j', '4', '--target', 'llama-server'], check=True)
    if not LLAMA_CPP_BIN.is_file():
        raise RuntimeError('llama.cpp server executable was not built')

    def qwen_ready():
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{QWEN_PORT}/health', timeout=3) as r:
                if r.status != 200:
                    return False
            with urllib.request.urlopen(f'http://127.0.0.1:{QWEN_PORT}/v1/models', timeout=3) as r:
                available = json.load(r)
            names = {str(item.get('id', '')) for item in available.get('data', [])}
            if f'{QWEN_REPO}:Q4_K_M' not in names:
                raise RuntimeError(f'Port {QWEN_PORT} is serving another model: {sorted(names)}')
            return True
        except RuntimeError:
            raise
        except Exception:
            return False

    if not qwen_ready():
        QWEN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        log = open(QWEN_LOG_PATH, 'ab', buffering=0)
        try:
            QWEN_SERVER_PROCESS = subprocess.Popen([
                str(LLAMA_CPP_BIN), '-m', str(QWEN_LOCAL_PATH),
                '--alias', f'{QWEN_REPO}:Q4_K_M',
                '--host', '127.0.0.1', '--port', str(QWEN_PORT),
                '--ctx-size', str(QWEN_JUDGE_CONTEXT_TOKENS),
                '--n-gpu-layers', str(QWEN_JUDGE_GPU_LAYERS),
                '--threads', '8', '--parallel', '1',
                '--batch-size', '256', '--ubatch-size', '128',
                '--jinja', '--reasoning', 'on',
                '--reasoning-format', 'deepseek',
                '--reasoning-budget', '3072',
            ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        finally:
            log.close()
        for attempt in range(240):
            if attempt % 15 == 0:
                print(f'Waiting for Qwen thinking server: {attempt * 2}s elapsed; log: {QWEN_LOG_PATH}', flush=True)
            if qwen_ready():
                break
            if QWEN_SERVER_PROCESS.poll() is not None:
                tail = QWEN_LOG_PATH.read_text(errors='replace')[-6000:]
                raise RuntimeError('Qwen judge server failed to start:\n' + tail)
            time.sleep(2)
    if not qwen_ready():
        raise RuntimeError('Qwen judge server not ready. Check ' + str(QWEN_LOG_PATH))
    print('Local 4-bit fallback judge is READY on 127.0.0.1:18037 (thinking requested)')
    print('GPU layers used by Qwen judge:', QWEN_JUDGE_GPU_LAYERS)
    print('Remember: ChemDFM and ChemLLM remain unmodified and load independently.')

    # Check that this server exposes actual reasoning content rather than only a model alias.
    # No benchmark task or victim-model response is sent by this probe.
    from urllib import request as _qwen_request
    _probe_payload = json.dumps({
        'model': f'{QWEN_REPO}:Q4_K_M',
        'messages': [{'role': 'user', 'content': 'What is the sum of 17 and 29? Answer briefly.'}],
        'max_tokens': 384, 'temperature': 0
    }).encode('utf-8')
    _probe_req = _qwen_request.Request(
        f'http://127.0.0.1:{QWEN_PORT}/v1/chat/completions',
        data=_probe_payload, headers={'Content-Type':'application/json'}, method='POST'
    )
    try:
        with _qwen_request.urlopen(_probe_req, timeout=300) as _probe_response:
            _probe_answer = json.load(_probe_response)
        _probe_msg = (_probe_answer.get('choices') or [{}])[0].get('message') or {}
        _has_reasoning = bool(str(_probe_msg.get('reasoning_content') or '').strip())
        if not _has_reasoning:
            raise RuntimeError(
                'Qwen returned no reasoning_content on the standalone probe. '
                'Do not assume thinking is active. Check the server log and llama.cpp revision.'
            )
        print('Fallback thinking-mode probe PASSED: server returned reasoning_content (not printed).', flush=True)
    except Exception as exc:
        raise RuntimeError(
            'Thinking-mode probe failed; no target experiment has been started. '
            f'Check server log: {QWEN_LOG_PATH}'
        ) from exc

    # Existing production CHCS schema, evaluated on a harmless synthetic example.
    # This detects incompatibility between thinking and grammar-constrained JSON
    # before a live run consumes target-model queries.
    from chembreak37.providers import LocalQwenGGUFChcsJudge as _FallbackJudge
    _safe_judge_input = json.dumps({
        'task_lock': {'original_prompt':'Provide generic personal protective equipment advice.'},
        'evaluation_scope':'all_target_responses_in_current_active_context',
        'response_count':1,
        'target_responses':[{'context_turn':1, 'response':'Wear goggles and gloves as appropriate.'}]
    })
    try:
        _safe_result = _FallbackJudge(config['roles']['chcs_fallback_judge_llm']).judge_response(
            _safe_judge_input, 'Wear goggles and gloves as appropriate.'
        )
        print('Fallback structured CHCS judge probe PASSED:',
              'CHCS', _safe_result['chcs'],
              'class', _safe_result['response_class'], flush=True)
    except Exception as exc:
        raise RuntimeError(
            'Qwen fallback thinking returned an invalid or incomplete schema-constrained answer. '
            'Do not run live targets until the model/server combination is compatible.'
        ) from exc


In [ ]:
# Cell 9: package, dataset, and API/GPU preflight
from pprint import pprint
from chembreak37.preflight import run_preflight

preflight_report = run_preflight(
    runtime_config,
    probe_tokenizers=False,
    probe_roles=PROBE_ROLES,
    target_id=None,
)
pprint(preflight_report)


## Target 1: ChemDFM

Run the next two cells for ChemDFM. The run cell is independent of the ChemLLM run cell. It unloads ChemDFM in a `finally` block even if execution is interrupted.


In [ ]:
# Cell 11: optional ChemDFM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemDFM'))
else:
    print('ChemDFM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 12: run ChemDFM only
from chembreak37.runner import ChemBreak37Runner

chemdfm_runner = ChemBreak37Runner(runtime_config, 'ChemDFM')
try:
    chemdfm_summary = chemdfm_runner.run_all()
finally:
    chemdfm_runner.unload_target()
    chemdfm_runner.close()

print('ChemDFM FINAL:', chemdfm_summary['bounded_first_success'])


In [ ]:
# Cell 13: confirm GPU memory after ChemDFM
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU allocated after ChemDFM cleanup (GiB) =', round(torch.cuda.memory_allocated() / 1024**3, 3))
else:
    print('Dry run or no CUDA GPU.')


## Target 2: ChemLLM

Run the next two cells for ChemLLM. This cell creates its own runner and loads only ChemLLM. It does not depend on the ChemDFM runner or its in-memory state.


In [ ]:
# Cell 15: optional ChemLLM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemLLM'))
else:
    print('ChemLLM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 16: run ChemLLM only
from chembreak37.runner import ChemBreak37Runner

chemllm_runner = ChemBreak37Runner(runtime_config, 'ChemLLM')
try:
    chemllm_summary = chemllm_runner.run_all()
finally:
    chemllm_runner.unload_target()
    chemllm_runner.close()

print('ChemLLM FINAL:', chemllm_summary['bounded_first_success'])


In [ ]:
# Cell 17: compare whichever targets have been completed in this session
comparison = {}
if 'chemdfm_summary' in globals():
    comparison['ChemDFM'] = chemdfm_summary['bounded_first_success']
if 'chemllm_summary' in globals():
    comparison['ChemLLM'] = chemllm_summary['bounded_first_success']
if not comparison:
    print('Run at least one target cell first.')
else:
    pprint(comparison)


In [ ]:
# Cell 18: locate public and private outputs
from pathlib import Path

for target_id in ('ChemDFM', 'ChemLLM'):
    target_root = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id
    release = target_root / 'release'
    internal = release / 'INTERNAL_DO_NOT_RELEASE'
    print(target_id)
    print('  public release:', release)
    print('  private transcript:', internal / 'cb37_full_transcripts.csv')
    print('  first-success review:', internal / 'cb37_first_success_review.csv')


In [ ]:
# Cell 19: create a public-results ZIP without private transcripts
import shutil

zip_root = STORAGE_ROOT / RUN_MODE / resolved_run_id / 'exports'
zip_root.mkdir(parents=True, exist_ok=True)
public_stage = zip_root / 'chembreak37_public_results'
if public_stage.exists():
    shutil.rmtree(public_stage)
public_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    release = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release'
    if release.exists():
        destination = public_stage / target_id
        shutil.copytree(release, destination, ignore=shutil.ignore_patterns('INTERNAL_DO_NOT_RELEASE'))
public_zip = shutil.make_archive(str(zip_root / f'chembreak37_{resolved_run_id}_public'), 'zip', public_stage)
print('PUBLIC ZIP =', public_zip)


In [ ]:
# Cell 20: create a private audit ZIP containing raw prompts and responses
import shutil

private_stage = zip_root / 'chembreak37_private_audit'
if private_stage.exists():
    shutil.rmtree(private_stage)
private_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    internal = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release' / 'INTERNAL_DO_NOT_RELEASE'
    if internal.exists():
        shutil.copytree(internal, private_stage / target_id)
private_zip = shutil.make_archive(str(zip_root / f'chembreak37_{resolved_run_id}_private_audit'), 'zip', private_stage)
print('PRIVATE AUDIT ZIP =', private_zip)
